In [1]:
import pandas as pd
import numpy as np

In [2]:
        
data = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/2022-12-SGFTFN_SPINELS_unprocessed.xlsx",header=0,index_col=0)
# data = pd.concat([data,data1],axis=0)
oxide = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/oxide_data.xlsx", sheet_name="Sheet1",index_col=0,header=0)
oxlist = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5"]

def wt_to_mol(data, oxide = oxide):
    data[data<=2] = 0
    oxlist1 = oxlist.copy()
    oxide = oxide.T[oxlist1].iloc[0, :].to_numpy()
    data = normalize(data)
    [r, c] = data.shape
    data_f = np.empty((r, c))
    for i in range(0, c):
        data_f[:, i] = data[:, i] / oxide[i]
    data
    data_f = normalize(data_f)
    return(data_f.round(2))


def normalize(data):
    [r, c] = data.shape
    a = data.sum(axis=1).reshape((len(data), 1))
    data_formatted = ((data*100)/a).round(1)
    return(data_formatted)


def cat_calc(data1,oxide_list):
    total = data1.columns.get_loc("Total")
    o_no = data1.loc[:,"Oxygen_no"]
    data = data1.iloc[:,:total].copy()
    oxide = oxide_list.loc[data.columns]
    data = data.div(oxide['Mol. Wt.'].values,axis=1).round(3)
    data = data.mul(oxide['O_no'].values,axis=1).round(3)
    norm = o_no.div(data.sum(axis=1)).round(3)
    data = data.mul(norm.values,axis=0).round(3)
    data = data.mul(oxide['Cat_per_o'].values,axis=1).round(3)
    total = data.sum(axis=1).round(3)
    data.columns = oxide['Cation'].values
    data['Cation_Total'] = total
    return data.round(3)


In [3]:
data2 = data[['SIO2(WT%)','TIO2(WT%)','AL2O3(WT%)','CR2O3(WT%)','FE2O3T(WT%)', 'FE2O3(WT%)', 'FEOT(WT%)','FEO(WT%)', 'MNO(WT%)','MGO(WT%)','CAO(WT%)', 'NA2O(WT%)','K2O(WT%)','P2O5(WT%)','MINERAL']]

In [4]:
data2.MINERAL.unique()

array(['SPINEL', 'MAGNETITE', 'CHROMITE', 'TITANO-MAGNETITE',
       'CHROME-SPINEL', 'HERCYNITE', 'ULVOSPINEL', 'MAGNESIOFERRITE',
       'PLEONASTE', 'AL-SPINEL', 'GAHNITE', 'FE-CHROMITE',
       'MAGNETITE/CHROMITE', 'JACOBSITE', 'PICOTITE', nan], dtype=object)

In [5]:
len(data2)

80651

In [6]:
data_cleaned = data2.loc[~data2['AL2O3(WT%)'].isna(),:]
# data_px = data_cleaned.loc[(data_cleaned['MINERAL']=="ILMENITE"),:]
data_px = data_cleaned.copy()
data_px.loc[~data_px['FEOT(WT%)'].isna(),:]
data_px.pop("FE2O3(WT%)")
data_px.pop("FE2O3T(WT%)")
data_px.pop("FEO(WT%)")
data_px.columns = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5",'Mineral']
mineral = data_px['Mineral']
data_px

,SiO2,TiO2,Al2O3,Cr2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,,
[36] CHEN CHANG-HWA (1992),0.29,1.67,15.28,38.63,33.75,0.25,9.84,0.05,NaN,NaN,NaN,SPINEL
[36] CHEN CHANG-HWA (1992),0.35,1.64,14.97,38.3,32.83,0.33,11.1,NaN,NaN,NaN,NaN,SPINEL
[36] CHEN CHANG-HWA (1992),0.29,1.64,14.8,38.79,35.7,0.26,8.24,0.08,NaN,NaN,NaN,SPINEL
[36] CHEN CHANG-HWA (1992),0.09,1.72,33.4,19.33,29.96,0.15,14.46,NaN,NaN,NaN,NaN,SPINEL
[36] CHEN CHANG-HWA (1992),0.11,1.39,30.7,25.3,28.15,0.15,14.36,NaN,NaN,NaN,NaN,SPINEL
...,...,...,...,...,...,...,...,...,...,...,...,...
[26127] DWIVEDI S. K. (2022),0,0.42,10.02,53.59,29.46,1.16,5.35,0.15,NaN,NaN,NaN,SPINEL
[26127] DWIVEDI S. K. (2022),0,0.35,10.69,53.6,26.65,0,6.82,0.11,NaN,NaN,NaN,SPINEL
[26127] DWIVEDI S. K. (2022),0,0.39,11.13,56.11,19.96,0.36,10.82,0.05,NaN,NaN,NaN,SPINEL


In [7]:
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')
data_px2 = data_px.fillna(0)
total = data_px2.sum(axis=1)
data_px2['Mineral'] = mineral
data_px2 = data_px2[(total>98) & (total<101)]
mineral = data_px2.pop("Mineral")
col = data_px2.columns
ind = data_px2.index
data_px2 = pd.DataFrame(wt_to_mol(data_px2.to_numpy(),oxide),columns = col,index=ind)
# data_px2 = data
data_px2

non_essential_sum = data_px2[['SiO2',"CaO", "Na2O", "K2O"]].sum(axis=1)
data_px2 = data_px2[non_essential_sum<1]
m = data_px2[["FeO","MnO","MgO"]].sum(axis=1)
data_px2 = data_px2[ (m >= 49) & (m <= 51)]
data_px2 = data_px2[(data_px2.Al2O3 + data_px2.Cr2O3 >= 49) & (data_px2.Al2O3 + data_px2.Cr2O3 <= 51)]
# data_px2['P2O5'] = 0
data_px2['Mineral'] = "Spl"
data_px2['Al2O3'] = data_px2['Al2O3'] + data_px2['Cr2O3']
data_px2.pop("Cr2O3")
data_px2
# data_px2.to_excel("/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/molar tables/new data/Spl_processed_mol.xlsx")


,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,
[919] KOGARKO L. N. (1995),0.0,0.0,49.1,17.4,0.0,33.5,0.0,0.0,0.0,0.0,Spl
[919] KOGARKO L. N. (1995),0.0,0.0,49.8,14.5,0.0,35.7,0.0,0.0,0.0,0.0,Spl
[1332] LUDDEN J. N. (1977),0.0,0.0,49.9,20.6,0.0,29.5,0.0,0.0,0.0,0.0,Spl
[2592] KRISHNAMURTHY P. (1988),0.0,0.0,49.7,11.5,0.0,38.7,0.0,0.0,0.0,0.0,Spl
[3131] BERRY R. F. (1981),0.0,0.0,49.0,14.2,0.0,36.8,0.0,0.0,0.0,0.0,Spl
...,...,...,...,...,...,...,...,...,...,...,...
[25978] CHEN BAO-YUN (2022),0.0,0.0,49.8,14.7,0.0,35.5,0.0,0.0,0.0,0.0,Spl
[25978] CHEN BAO-YUN (2022),0.0,0.0,49.8,14.7,0.0,35.5,0.0,0.0,0.0,0.0,Spl
[25978] CHEN BAO-YUN (2022),0.0,0.0,49.6,19.4,0.0,31.0,0.0,0.0,0.0,0.0,Spl


In [8]:
data_px2.sort_values("TiO2",ascending=False)

,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,
[919] KOGARKO L. N. (1995),0.0,0.0,49.1,17.4,0.0,33.5,0.0,0.0,0.0,0.0,Spl
[18789] CHEN SHENG-SHENG (2012),0.0,0.0,49.5,8.8,0.0,41.7,0.0,0.0,0.0,0.0,Spl
[18352] MARTIN A. P. (2014),0.0,0.0,50.4,16.1,0.0,33.5,0.0,0.0,0.0,0.0,Spl
[18352] MARTIN A. P. (2014),0.0,0.0,49.2,12.6,0.0,38.2,0.0,0.0,0.0,0.0,Spl
[18352] MARTIN A. P. (2014),0.0,0.0,49.2,12.3,0.0,38.5,0.0,0.0,0.0,0.0,Spl
...,...,...,...,...,...,...,...,...,...,...,...
[16480] BAPTISTE V. (2012),0.0,0.0,50.1,18.0,0.0,31.9,0.0,0.0,0.0,0.0,Spl
[16480] BAPTISTE V. (2012),0.0,0.0,50.3,17.7,0.0,32.1,0.0,0.0,0.0,0.0,Spl
[16480] BAPTISTE V. (2012),0.0,0.0,50.3,18.3,0.0,31.4,0.0,0.0,0.0,0.0,Spl


In [43]:
data_cleaned = data2.loc[~data2['FEO(WT%)'].isna(),:]
# data_px = data_cleaned[(data_cleaned["MINERAL"]=="MAGNETITE") | (data_cleaned["MINERAL"]=="TITANO-MAGNETITE")]
data_px = data_cleaned.copy()
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')

cond = (data_px['FEOT(WT%)'].isna()) & (~data_px['FE2O3T(WT%)'].isna())
data_px.loc[cond,'FEOT(WT%)'] = data_px.loc[cond,'FE2O3T(WT%)']*.8998
cond = (data_px['FEOT(WT%)'].isna()) & (data_px['FE2O3T(WT%)'].isna())
data_px.loc[cond,'FEOT(WT%)'] = data_px.loc[cond,'FEO(WT%)'] + (data_px.loc[cond,'FE2O3(WT%)']*.8998)
data_px.loc[~data_px['FEOT(WT%)'].isna(),:]
data_px.pop("FE2O3(WT%)")
data_px.pop("FE2O3T(WT%)")
data_px.pop("FEO(WT%)")
data_px['Mineral'] = data_cleaned['MINERAL']
data_px.columns = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5",'Mineral']
mineral = data_px['Mineral']
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')
data_px2 = data_px.fillna(0)
total = data_px2.sum(axis=1)
data_px2['Mineral'] = mineral
data_px2 = data_px2[(total>=95) & (total<=102)]
mineral = data_px2.pop("Mineral")
col = data_px2.columns
ind = data_px2.index
data_px2 = pd.DataFrame(wt_to_mol(data_px2.to_numpy(),oxide),columns = col,index=ind)
# data_px2 = data

non_essential_sum = data_px2[['SiO2','Cr2O3',"Al2O3","MnO","MgO","CaO", "Na2O", "K2O"]].sum(axis=1)
data_px2 = data_px2[non_essential_sum<2]
m = data_px2["FeO"]
data_px2 = data_px2[ (m >= 89) & (m <= 100)]
# data_px2 = data_px2[(data_px2.Al2O3 + data_px2.Cr2O3 >= 49) & (data_px2.Al2O3 + data_px2.Cr2O3 <= 51)]
# data_px2['P2O5'] = 0
data_px2['Mineral'] = "Mag/Hem"
data_px2['Al2O3'] = data_px2['Al2O3'] + data_px2['Cr2O3']
data_px2.pop("Cr2O3")
data_px2.to_excel("/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/molar tables/new data/mag_processed_mol.xlsx")
data_px2.sort_values("Al2O3",ascending=False)


,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,
[17055] BARKER S. J. (2013),0.0,8.8,1.9,89.3,0.0,0.0,0.0,0.0,0.0,0.0,Mag/Hem
[19087] GAO JIANFENG (2015),0.0,7.5,1.9,90.6,0.0,0.0,0.0,0.0,0.0,0.0,Mag/Hem
[19087] GAO JIANFENG (2015),0.0,9.1,1.9,89.0,0.0,0.0,0.0,0.0,0.0,0.0,Mag/Hem
[19087] GAO JIANFENG (2015),0.0,9.1,1.9,89.1,0.0,0.0,0.0,0.0,0.0,0.0,Mag/Hem
[17624] WATERS L. E. (2013),0.0,7.2,1.9,90.8,0.0,0.0,0.0,0.0,0.0,0.0,Mag/Hem
...,...,...,...,...,...,...,...,...,...,...,...
[18384] MARCAIDA M. (2014),0.0,9.7,0.0,90.3,0.0,0.0,0.0,0.0,0.0,0.0,Mag/Hem
[18384] MARCAIDA M. (2014),0.0,10.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,Mag/Hem
[18384] MARCAIDA M. (2014),0.0,10.6,0.0,89.4,0.0,0.0,0.0,0.0,0.0,0.0,Mag/Hem


In [ ]:
data_cleaned[(data_cleaned["MINERAL"]=="MAGNETITE") | (data_cleaned[
"MINERAL"]=="TITANO-MAGNETITE")]

In [ ]:
data_cleaned[~data_cleaned['FE2O3T(WT%)'].isna(),:]